In [3]:
!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.5 MB/s eta 0:00:00


In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Hn49rXIC9iN2cSuJUOH7")
project = rf.workspace("marblerace").project("skin-diseases-ewn74")
version = project.version(2)
dataset = version.download("folder")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to skin-diseases-2 in folder:: 100%|██████████| 4108/4108 [00:00<00:00, 5910.86it/s]


In [5]:
import os
for split in ["train", "valid", "test"]:
    print(f"\n{split}:")
    split_path = f"{dataset.location}/{split}"
    for cls in os.listdir(split_path):
        n = len(os.listdir(f"{split_path}/{cls}"))
        print(f"  {cls}: {n} images")


train:
  Vascular Lesion: 202 images
  Squamous Cell Carcinoma: 290 images
  Seborrheic Keratosis: 346 images
  Dermatofibroma: 398 images
  Pigmented Benign Keratosis: 345 images
  Basal Cell Carcinoma: 498 images
  Acitinic Keratosis: 431 images
  Melanoma: 377 images
  Nevus: 349 images

valid:
  Vascular Lesion: 58 images
  Squamous Cell Carcinoma: 83 images
  Seborrheic Keratosis: 115 images
  Pigmented Benign Keratosis: 148 images
  Acitinic Keratosis: 63 images
  Melanoma: 128 images
  Nevus: 150 images

test:
  Vascular Lesion: 29 images
  Squamous Cell Carcinoma: 41 images
  Seborrheic Keratosis: 33 images


In [6]:
import os, shutil
from sklearn.model_selection import train_test_split

src = dataset.location
classes = os.listdir(f"{src}/train")

# collect all images per class across all existing splits
all_images = {cls: [] for cls in classes}
for split in ["train", "valid", "test"]:
    split_path = f"{src}/{split}"
    if not os.path.exists(split_path):
        continue
    for cls in os.listdir(split_path):
        folder = f"{split_path}/{cls}"
        for img in os.listdir(folder):
            all_images.setdefault(cls, []).append(f"{folder}/{img}")

# build new stratified split
new_root = "/content/skin_data_resplit"
for split in ["train", "valid", "test"]:
    for cls in classes:
        os.makedirs(f"{new_root}/{split}/{cls}", exist_ok=True)

for cls, imgs in all_images.items():
    train_imgs, temp_imgs = train_test_split(imgs, test_size=0.30, random_state=42)
    valid_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)

    for img in train_imgs:
        shutil.copy(img, f"{new_root}/train/{cls}/{os.path.basename(img)}")
    for img in valid_imgs:
        shutil.copy(img, f"{new_root}/valid/{cls}/{os.path.basename(img)}")
    for img in test_imgs:
        shutil.copy(img, f"{new_root}/test/{cls}/{os.path.basename(img)}")

print("Re-split complete. New dataset at:", new_root)

Re-split complete. New dataset at: /content/skin_data_resplit


In [7]:
for split in ["train", "valid", "test"]:
    print(f"\n{split}:")
    for cls in os.listdir(f"{new_root}/{split}"):
        n = len(os.listdir(f"{new_root}/{split}/{cls}"))
        print(f"  {cls}: {n} images")


train:
  Vascular Lesion: 202 images
  Squamous Cell Carcinoma: 289 images
  Seborrheic Keratosis: 345 images
  Dermatofibroma: 278 images
  Pigmented Benign Keratosis: 345 images
  Basal Cell Carcinoma: 348 images
  Acitinic Keratosis: 345 images
  Melanoma: 353 images
  Nevus: 349 images

valid:
  Vascular Lesion: 43 images
  Squamous Cell Carcinoma: 62 images
  Seborrheic Keratosis: 74 images
  Dermatofibroma: 60 images
  Pigmented Benign Keratosis: 74 images
  Basal Cell Carcinoma: 75 images
  Acitinic Keratosis: 74 images
  Melanoma: 76 images
  Nevus: 75 images

test:
  Vascular Lesion: 44 images
  Squamous Cell Carcinoma: 63 images
  Seborrheic Keratosis: 75 images
  Dermatofibroma: 60 images
  Pigmented Benign Keratosis: 74 images
  Basal Cell Carcinoma: 75 images
  Acitinic Keratosis: 75 images
  Melanoma: 76 images
  Nevus: 75 images


In [8]:
from ultralytics import YOLO

model = YOLO("yolov8s-cls.pt")
results = model.train(
    data=new_root,
    epochs=100,
    imgsz=224,
    batch=32,
    patience=40,
    project="skin_disease",
    name="baseline"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/skin_data_resplit, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, frac

In [9]:
from ultralytics import YOLO
model = YOLO("yolov8m-cls.pt")
model.train(
    data="/content/skin_data_resplit",
    epochs=100, imgsz=224, batch=32, patience=30,
    project="skin_disease", name="v3_medium"
)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/skin_data_resplit, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v3_medium, nbs=64, nms=False, opset=None, optimize=

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f8d1724cdd0>
curves: []
curves_results: []
fitness: 0.8678629696369171
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.7585644125938416, 'metrics/accuracy_top5': 0.9771615266799927, 'fitness': 0.8678629696369171}
save_dir: PosixPath('/content/runs/classify/skin_disease/v3_medium')
speed: {'preprocess': 0.20534200652345422, 'inference': 1.5820073539948998, 'loss': 0.00018061500831613876, 'postprocess': 0.018640533443584014}
top1: 0.7585644125938416
top5: 0.9771615266799927